# 🧠 การเพิ่มประสิทธิภาพด้วย Gradient Descent

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Gradient Descent**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายคณิตศาสตร์หลักของการอัปเดตเกรเดียนต์และบทบาทของอัตราการเรียนรู้ (Learning Rate)
2. อิมพลีเมนต์ 1D gradient descent สำหรับพื้นที่ผิวแบบนูน (convex) และแบบไม่นูน (non-convex) เพื่อดูว่าค่าเริ่มต้น (initialization) ส่งผลต่อการลู่เข้าสู่จุดต่ำสุดเฉพาะที่ (local minima) เทียบกับจุดต่ำสุดรวม (global minima) อย่างไร
3. แสดงภาพผลลัพธ์ของอัตราการเรียนรู้ (learning rates) ที่แตกต่างกัน (เล็กเกินไป, เหมาะสม, ใหญ่เกินไป) ต่อเส้นทางการเคลื่อนที่ลง (descent trajectories)
4. อิมพลีเมนต์ **2D Gradient Descent optimizer จากศูนย์ (from scratch)** เพื่อหาค่าต่ำสุดของ $f(x,y) = x^2 + 3y^2$
5. พล็อตเส้นทางการเพิ่มประสิทธิภาพบนเส้นชั้นความสูงแบบ 2 มิติ (2D contour lines) เพื่อดูภาพการลู่เข้า
6. เชื่อมโยงแนวคิดเหล่านี้กับการอัปเดตพารามิเตอร์การเรียนรู้เชิงลึกใน YOLO

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลย

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. 1D Convex Optimization: $f(x) = x^2$

มาตรวจสอบว่าอัตราการเรียนรู้ ($\alpha$) ส่งผลต่อการลู่เข้าอย่างไร
เราเปรียบเทียบอัตราการเรียนรู้สามแบบดังนี้:
1.  **Too Small ($\alpha = 0.02$):** Converges very slowly.
2.  **Optimal ($\alpha = 0.15$):** Converges smoothly and quickly.
3.  **Too Large ($\alpha = 1.05$):** Oscillates, overshoots, and diverges!

In [ ]:
def f_convex(x):
    return x ** 2

def df_convex(x):
    return 2 * x

def run_gd_1d(x_start, lr, epochs=15):
    x = x_start
    history = [x]
    for _ in range(epochs):
        grad = df_convex(x)
        x = x - lr * grad
        history.append(x)
    return np.array(history)

# Run simulations
hist_small = run_gd_1d(10.0, 0.02)
hist_opt = run_gd_1d(10.0, 0.15)
hist_large = run_gd_1d(10.0, 1.05)

# Plot trajectories
x_arr = np.linspace(-12, 12, 100)
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.plot(x_arr, f_convex(x_arr), color='gray')
plt.scatter(hist_small, f_convex(hist_small), color='red', zorder=5)
plt.plot(hist_small, f_convex(hist_small), color='red', linestyle='-')
plt.title('Small LR (α = 0.02): Slow Descent')
plt.xlabel('x')
plt.ylabel('Cost')

plt.subplot(1, 3, 2)
plt.plot(x_arr, f_convex(x_arr), color='gray')
plt.scatter(hist_opt, f_convex(hist_opt), color='green', zorder=5)
plt.plot(hist_opt, f_convex(hist_opt), color='green', linestyle='-')
plt.title('Optimal LR (α = 0.15): Fast Convergence')
plt.xlabel('x')

plt.subplot(1, 3, 3)
plt.plot(x_arr, f_convex(x_arr), color='gray')
plt.scatter(hist_large, f_convex(hist_large), color='purple', zorder=5)
plt.plot(hist_large, f_convex(hist_large), color='purple', linestyle='-')
plt.title('Large LR (α = 1.05): Exploding/Overshooting')
plt.xlabel('x')

plt.tight_layout()
plt.show()

## 2. Non-Convex Landscape and Local Minima

ลองมาเพิ่มประสิทธิภาพฟังก์ชันแบบไม่นูน $f(x) = x^4 - 3x^3 + 2$ พื้นที่ผิวนี้มีทั้งจุดต่ำสุดเฉพาะที่ (local minimum) และจุดต่ำสุดรวม (global minimum) การเริ่มต้นที่พิกัดต่างกันจะนำโมเดลไปสู่การลู่เข้าหาจุดต่ำสุดที่ต่างกัน!

In [ ]:
def f_nonconvex(x):
    return x**4 - 3*x**3 + 2

def df_nonconvex(x):
    return 4*x**3 - 9*x**2

def run_gd_nonconvex(x_start, lr=0.05, epochs=30):
    x = x_start
    history = [x]
    for _ in range(epochs):
        grad = df_nonconvex(x)
        x = x - lr * grad
        history.append(x)
    return np.array(history)

# Simulation starting at x = -0.8 vs. x = 3.0
hist_left = run_gd_nonconvex(-0.8)
hist_right = run_gd_nonconvex(3.0)

x_arr = np.linspace(-1.5, 3.5, 100)
plt.figure(figsize=(10, 6))
plt.plot(x_arr, f_nonconvex(x_arr), color='black', linewidth=2, label='Cost Function f(x)')
plt.plot(hist_left, f_nonconvex(hist_left), color='red', marker='o', label='Trajectory 1: Trap in local min')
plt.plot(hist_right, f_nonconvex(hist_right), color='green', marker='s', label='Trajectory 2: Global min')
plt.xlabel('x')
plt.ylabel('Cost')
plt.title('Gradient Descent on a Non-Convex Landscape')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. 2D Gradient Descent from Scratch

ตอนนี้เรามาอิมพลีเมนต์ Gradient Descent ในแบบ 2 มิติ เพื่อหาค่าต่ำสุดของพาราโบโลอิด (Paraboloid):
$$f(x, y) = x^2 + 3y^2$$

เกรเดียนต์:
$$\frac{\partial f}{\partial x} = 2x, \quad \frac{\partial f}{\partial y} = 6y$$

In [ ]:
def f_2d(x, y):
    return x**2 + 3*y**2

def grad_2d(x, y):
    return np.array([2*x, 6*y])

def gradient_descent_2d(start_pos, lr, epochs=30):
    pos = np.array(start_pos)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_2d(pos[0], pos[1])
        pos = pos - lr * grad
        history.append(pos.copy())
    return np.array(history)

# Run 2D optimization
start_point = [8.0, 8.0]
history_2d = gradient_descent_2d(start_point, lr=0.1)

# Plotting the 2D Contour Map
x = np.linspace(-10, 10, 100)
y = np.linspace(-10, 10, 100)
X, Y = np.meshgrid(x, y)
Z = f_2d(X, Y)

plt.figure(figsize=(8, 7))
contours = plt.contour(X, Y, Z, levels=20, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)
plt.plot(history_2d[:, 0], history_2d[:, 1], color='red', marker='o', linewidth=2, label='Descent Path')
plt.scatter(0, 0, color='blue', s=100, marker='*', zorder=5, label='Global Minimum')
plt.xlabel('x')
plt.ylabel('y')
plt.title('2D Gradient Descent Trajectory on f(x,y) = x² + 3y²')
plt.legend()
plt.show()

## 💡 ความเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **การเพิ่มประสิทธิภาพพารามิเตอร์ (Parameter Optimization):** ในระหว่างการฝึกของ YOLO โมเดลจะมีน้ำหนัก (weights) หลายล้านค่า ในแต่ละขั้นตอนการฝึก จะมีการคำนวณค่าสูญเสียหรือลอส (loss) (เช่น ข้อผิดพลาดการทำนายกรอบ Bounding Box + ข้อผิดพลาดการจำแนกประเภท Classification) และการแพร่ย้อนกลับ (**Backpropagation**) จะประเมินค่าเกรเดียนต์ (อนุพันธ์ย่อย) ของลอสเมื่อเทียบกับน้ำหนักทุกๆ ตัวในโครงข่าย
*   **SGD และ Adam:** แทนที่จะคำนวณเกรเดียนต์จากชุดข้อมูลทั้งหมด (ซึ่งมีขนาดใหญ่มาก) เราจะคำนวณเกรเดียนต์จากกลุ่มตัวอย่างขนาดเล็กที่เรียกว่า **mini-batches** (Stochastic Gradient Descent) ตัวปรับค่าหรือออพติไมเซอร์สมัยใหม่ (Modern optimizers) ยังเพิ่ม **แรงส่ง (momentum)** (ค่าเฉลี่ยเคลื่อนที่ของเกรเดียนต์ก่อนหน้า) หรือปรับอัตราการเรียนรู้ตามพารามิเตอร์แต่ละตัว (เช่น Adam) เพื่อหลีกเลี่ยงจุดอานม้า (saddle points) และเร่งการลู่เข้าให้เร็วขึ้น